In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [2]:
movie_ds=pd.read_csv('../movie_rec/imdb_movies.csv')
movie_ds['names']=movie_ds['names'].str.lower()

In [3]:
movie_ds.isnull().sum()
moive_ds=movie_ds.dropna().reset_index(drop=True)
movie_ds=movie_ds.drop_duplicates().reset_index(drop=True)


In [4]:
movie_ds

,names,date_x,score,genre,overview,crew,orig_title,status,orig_lang,budget_x,revenue,country
0,creed iii,03/02/2023,73.0,"Drama, Action","After dominating the boxing world, Adonis Cree...","Michael B. Jordan, Adonis Creed, Tessa Thompso...",Creed III,Released,English,75000000.0,2.716167e+08,AU
1,avatar: the way of water,12/15/2022,78.0,"Science Fiction, Adventure, Action",Set more than a decade after the events of the...,"Sam Worthington, Jake Sully, Zoe Saldaña, Neyt...",Avatar: The Way of Water,Released,English,460000000.0,2.316795e+09,AU
2,the super mario bros. movie,04/05/2023,76.0,"Animation, Adventure, Family, Fantasy, Comedy","While working underground to fix a water main,...","Chris Pratt, Mario (voice), Anya Taylor-Joy, P...",The Super Mario Bros. Movie,Released,English,100000000.0,7.244590e+08,AU
3,mummies,01/05/2023,70.0,"Animation, Comedy, Family, Adventure, Fantasy","Through a series of unfortunate events, three ...","Óscar Barberán, Thut (voice), Ana Esther Albor...",Momias,Released,"Spanish, Castilian",12300000.0,3.420000e+07,AU
4,supercell,03/17/2023,61.0,Action,Good-hearted teenager William always lived in ...,"Skeet Ulrich, Roy Cameron, Anne Heche, Dr Quin...",Supercell,Released,English,77000000.0,3.409420e+08,US
...,...,...,...,...,...,...,...,...,...,...,...,...
10173,20th century women,12/28/2016,73.0,Drama,"In 1979 Santa Barbara, California, Dorothea Fi...","Annette Bening, Dorothea Fields, Lucas Jade Zu...",20th Century Women,Released,English,7000000.0,9.353729e+06,US
10174,delta force 2: the colombian connection,08/24/1990,54.0,Action,When DEA agents are taken captive by a ruthles...,"Chuck Norris, Col. Scott McCoy, Billy Drago, R...",Delta Force 2: The Colombian Connection,Released,English,9145817.8,6.698361e+06,US
10175,the russia house,12/21/1990,61.0,"Drama, Thriller, Romance","Barley Scott Blair, a Lisbon-based editor of R...","Sean Connery, Bartholomew 'Barley' Scott Blair...",The Russia House,Released,English,21800000.0,2.299799e+07,US
10176,darkman ii: the return of durant,07/11/1995,55.0,"Action, Adventure, Science Fiction, Thriller, ...",Darkman and Durant return and they hate each o...,"Larry Drake, Robert G. Durant, Arnold Vosloo, ...",Darkman II: The Return of Durant,Released,English,116000000.0,4.756613e+08,US


In [5]:
cols=["date_x","score","orig_title","status","budget_x","orig_lang","revenue","country"]
movie_ds['genre']=movie_ds['genre'].str.replace('\xa0','',regex=False)
movie_ds=movie_ds.drop(cols,axis='columns')
movie_wo_crew=moive_ds

In [6]:
movie_ds

,names,genre,overview,crew
0,creed iii,"Drama,Action","After dominating the boxing world, Adonis Cree...","Michael B. Jordan, Adonis Creed, Tessa Thompso..."
1,avatar: the way of water,"Science Fiction,Adventure,Action",Set more than a decade after the events of the...,"Sam Worthington, Jake Sully, Zoe Saldaña, Neyt..."
2,the super mario bros. movie,"Animation,Adventure,Family,Fantasy,Comedy","While working underground to fix a water main,...","Chris Pratt, Mario (voice), Anya Taylor-Joy, P..."
3,mummies,"Animation,Comedy,Family,Adventure,Fantasy","Through a series of unfortunate events, three ...","Óscar Barberán, Thut (voice), Ana Esther Albor..."
4,supercell,Action,Good-hearted teenager William always lived in ...,"Skeet Ulrich, Roy Cameron, Anne Heche, Dr Quin..."
...,...,...,...,...
10173,20th century women,Drama,"In 1979 Santa Barbara, California, Dorothea Fi...","Annette Bening, Dorothea Fields, Lucas Jade Zu..."
10174,delta force 2: the colombian connection,Action,When DEA agents are taken captive by a ruthles...,"Chuck Norris, Col. Scott McCoy, Billy Drago, R..."
10175,the russia house,"Drama,Thriller,Romance","Barley Scott Blair, a Lisbon-based editor of R...","Sean Connery, Bartholomew 'Barley' Scott Blair..."
10176,darkman ii: the return of durant,"Action,Adventure,Science Fiction,Thriller,Horror",Darkman and Durant return and they hate each o...,"Larry Drake, Robert G. Durant, Arnold Vosloo, ..."


In [7]:
base = movie_ds.dropna().drop_duplicates().reset_index(drop=True)
base['names'] = base['names'].str.lower()
base['genre'] = base['genre'].str.replace('\xa0', '', regex=False)

movie_ds = base.copy()
movie_ds['details'] = movie_ds['overview'] + " " + movie_ds['genre'] + " " + movie_ds['crew']

movie_wo_crew = base.copy()
movie_wo_crew['details'] = movie_wo_crew['overview'] + " " + movie_wo_crew['genre']

In [8]:
movie_ds=movie_ds.drop(["genre","crew","overview"],axis='columns')

In [9]:
movie_wo_crew.isnull().sum()

names       0
genre       0
overview    0
crew        0
details     0
dtype: int64

In [10]:
movie_wo_crew=movie_wo_crew.drop(["genre","crew","overview"],axis='columns')
movie_wo_crew

,names,details
0,creed iii,"After dominating the boxing world, Adonis Cree..."
1,avatar: the way of water,Set more than a decade after the events of the...
2,the super mario bros. movie,"While working underground to fix a water main,..."
3,mummies,"Through a series of unfortunate events, three ..."
4,supercell,Good-hearted teenager William always lived in ...
...,...,...
9868,20th century women,"In 1979 Santa Barbara, California, Dorothea Fi..."
9869,delta force 2: the colombian connection,When DEA agents are taken captive by a ruthles...
9870,the russia house,"Barley Scott Blair, a Lisbon-based editor of R..."
9871,darkman ii: the return of durant,Darkman and Durant return and they hate each o...


In [11]:
idx = movie_wo_crew[movie_wo_crew['names']=='spider-man'].index[0]
print(movie_wo_crew.loc[idx, 'details'])

After being bitten by a genetically altered spider at Oscorp, nerdy but endearing high school student Peter Parker is endowed with amazing powers to become the superhero known as Spider-Man. Fantasy,Action


In [12]:
idx = movie_ds[movie_ds['names']=='spider-man'].index[0]
print(movie_ds.loc[idx, 'details'])

After being bitten by a genetically altered spider at Oscorp, nerdy but endearing high school student Peter Parker is endowed with amazing powers to become the superhero known as Spider-Man. Fantasy,Action Tobey Maguire, Spider-Man / Peter Parker, Willem Dafoe, Green Goblin / Norman Osborn, Kirsten Dunst, Mary Jane Watson, James Franco, Harry Osborn, Cliff Robertson, Ben Parker, Rosemary Harris, May Parker, J.K. Simmons, J. Jonah Jameson, Joe Manganiello, Flash Thompson, Gerry Becker, Maximilian Fargas


In [13]:
tfidf=TfidfVectorizer(stop_words='english',max_features=10178)
tfidf_matrix=tfidf.fit_transform(movie_ds['details'])
tfidf_wo_crew=tfidf.fit_transform(movie_wo_crew['details'])
cv=CountVectorizer(stop_words='english',max_features=10000)
cv_matrix=cv.fit_transform(movie_ds['details'])
cv_wo_crew=cv.fit_transform(movie_wo_crew['details'])

In [14]:
print(f"Shape of the matrix: {tfidf_matrix.shape}")
print(f"Shape of the matrix: {tfidf_wo_crew.shape}")

Shape of the matrix: (9873, 10178)
Shape of the matrix: (9873, 10178)


In [15]:
print(f"Shape of the matrix: {cv_matrix.shape}")
print(cv_matrix)


Shape of the matrix: (9873, 10000)
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 431483 stored elements and shape (9873, 10000)>
  Coords	Values
  (0, 1061)	2
  (0, 9822)	1
  (0, 2007)	4
  (0, 1338)	1
  (0, 3034)	1
  (0, 5282)	1
  (0, 1557)	1
  (0, 3340)	1
  (0, 7118)	1
  (0, 2117)	3
  (0, 351)	2
  (0, 7495)	1
  (0, 8072)	1
  (0, 5385)	1
  (0, 8042)	1
  (0, 7105)	1
  (0, 2638)	1
  (0, 7154)	1
  (0, 8206)	1
  (0, 7579)	1
  (0, 3000)	1
  (0, 3342)	1
  (0, 4743)	1
  (0, 3145)	1
  (0, 8078)	1
  :	:
  (9872, 5433)	1
  (9872, 5268)	1
  (9872, 2533)	1
  (9872, 1593)	1
  (9872, 6957)	1
  (9872, 7099)	2
  (9872, 2300)	2
  (9872, 6438)	1
  (9872, 5908)	1
  (9872, 1532)	4
  (9872, 4690)	1
  (9872, 8269)	1
  (9872, 852)	1
  (9872, 5451)	1
  (9872, 9516)	1
  (9872, 6532)	2
  (9872, 3109)	1
  (9872, 2402)	1
  (9872, 8430)	1
  (9872, 8478)	1
  (9872, 3837)	1
  (9872, 3039)	1
  (9872, 3593)	1
  (9872, 255)	1
  (9872, 1817)	1


In [16]:
similar_tf=cosine_similarity(tfidf_matrix)
similar_tf


array([[1.        , 0.01241887, 0.01592963, ..., 0.00521587, 0.00176774,
        0.00217975],
       [0.01241887, 1.        , 0.00582277, ..., 0.021458  , 0.03558905,
        0.00310078],
       [0.01592963, 0.00582277, 1.        , ..., 0.02465392, 0.00764872,
        0.11386079],
       ...,
       [0.00521587, 0.021458  , 0.02465392, ..., 1.        , 0.0128604 ,
        0.        ],
       [0.00176774, 0.03558905, 0.00764872, ..., 0.0128604 , 1.        ,
        0.01094387],
       [0.00217975, 0.00310078, 0.11386079, ..., 0.        , 0.01094387,
        1.        ]], shape=(9873, 9873))

In [17]:
similar_tf_wo_crew=cosine_similarity(tfidf_wo_crew)
similar_tf_wo_crew

array([[1.        , 0.02608368, 0.01087189, ..., 0.00272918, 0.00307679,
        0.00491326],
       [0.02608368, 1.        , 0.01374005, ..., 0.        , 0.02864572,
        0.00764612],
       [0.01087189, 0.01374005, 1.        , ..., 0.0325406 , 0.00502819,
        0.02219623],
       ...,
       [0.00272918, 0.        , 0.0325406 , ..., 1.        , 0.00429206,
        0.        ],
       [0.00307679, 0.02864572, 0.00502819, ..., 0.00429206, 1.        ,
        0.02235024],
       [0.00491326, 0.00764612, 0.02219623, ..., 0.        , 0.02235024,
        1.        ]], shape=(9873, 9873))

In [18]:
similar=cosine_similarity(cv_matrix)
similar

array([[1.        , 0.04194676, 0.04164498, ..., 0.02796451, 0.01257489,
        0.01131407],
       [0.04194676, 1.        , 0.02166121, ..., 0.03636364, 0.13081399,
        0.01471225],
       [0.04164498, 0.02166121, 1.        , ..., 0.04332243, 0.01948093,
        0.28920677],
       ...,
       [0.02796451, 0.03636364, 0.04332243, ..., 1.        , 0.04905525,
        0.        ],
       [0.01257489, 0.13081399, 0.01948093, ..., 0.04905525, 1.        ,
        0.0132314 ],
       [0.01131407, 0.01471225, 0.28920677, ..., 0.        , 0.0132314 ,
        1.        ]], shape=(9873, 9873))

In [19]:
similar_cv_wo_crew=cosine_similarity(cv_wo_crew)
similar_cv_wo_crew

array([[1.        , 0.09128709, 0.05189993, ..., 0.02868877, 0.02594996,
        0.02868877],
       [0.09128709, 1.        , 0.07106691, ..., 0.        , 0.14213381,
        0.03928371],
       [0.05189993, 0.07106691, 1.        , ..., 0.06700252, 0.03030303,
        0.10050378],
       ...,
       [0.02868877, 0.        , 0.06700252, ..., 1.        , 0.03350126,
        0.        ],
       [0.02594996, 0.14213381, 0.03030303, ..., 0.03350126, 1.        ,
        0.03350126],
       [0.02868877, 0.03928371, 0.10050378, ..., 0.        , 0.03350126,
        1.        ]], shape=(9873, 9873))

In [20]:
similar_list=sorted(list(enumerate(similar_tf_wo_crew[74])),reverse=True, key= lambda tfidf_matrix:tfidf_matrix[1])
for i in similar_list[0:6]:
    print(movie_wo_crew.iloc[i[0]].names)

terrifier 2
terrifier
terrifier
clownhouse
paranormal activity: the ghost dimension
marrowbone


In [21]:
similar_list=sorted(list(enumerate(similar_tf[76])),reverse=True, key= lambda tfidf_matrix:tfidf_matrix[1])
for i in similar_list[1:6]:
    print(movie_ds.iloc[i[0]].names)

spider-man: homecoming
spider-man: far from home
doctor strange in the multiverse of madness
spider-man
doctor strange


In [22]:
similar_list=sorted(list(enumerate(similar_cv_wo_crew[76])),reverse=True, key= lambda cv_matrix:cv_matrix[1])
for i in similar_list[1:6]:
    print(movie_wo_crew.iloc[i[0]].names)

spider-man: into the spider-verse
spider-man: homecoming
spider-man: far from home
iboy
the matrix resurrections


In [23]:
similar_list=sorted(list(enumerate(similar[76])),reverse=True, key= lambda cv_matrix:cv_matrix[1])
for i in similar_list[1:6]:
    print(movie_ds.iloc[i[0]].names)

spider-man: homecoming
spider-man: far from home
doctor strange in the multiverse of madness
spider-man
doctor strange


In [24]:
def recommend_wo_crew(movies):
    movie_no_crew=movies.lower()
    movie_nc_idx=movie_wo_crew[movie_wo_crew['names']==movie_no_crew].index[0]
    similar_list_nc=sorted(list(enumerate(similar_tf_wo_crew[movie_nc_idx])),reverse=True,key=lambda tfnc:tfnc[1])
    similar_list_nc_cv=sorted(list(enumerate(similar_cv_wo_crew[movie_nc_idx])),reverse=True, key=lambda cvnc:cvnc[1])
    for i in similar_list_nc[1:6]:
        print(f"TF list :{movie_wo_crew.iloc[i[0]].names}")
    for i in similar_list_nc_cv[1:6]:
        print(f"CV list: {movie_wo_crew.iloc[i[0]].names}")

In [25]:
def recommend_tf(movies):
    movies=movies.lower()
    movies_index=movie_ds[movie_ds['names']==movies].index[0]
    similar_list=sorted(list(enumerate(similar_tf[movies_index])),reverse=True, key= lambda tf_matrix:tf_matrix[1])
    for i in similar_list[1:6]:
        print(movie_ds.iloc[i[0]].names)

In [26]:
def recommend(movies):
    movies=movies.lower()
    movies_index=movie_ds[movie_ds['names']==movies].index[0]
    similar_list=sorted(list(enumerate(similar[movies_index])),reverse=True, key= lambda cv_matrix:cv_matrix[1])
    for i in similar_list[1:6]:
        print(movie_ds.iloc[i[0]].names)

In [27]:
recommend("The amazing spider-man")

the amazing spider-man 2
spider-man
spider-man 2
spider-man 3
spider-man


In [28]:
recommend_tf("The amazing spider-man")

the amazing spider-man 2
spider-man 2
spider-man
spider-man 3
spider-man


In [29]:
recommend_wo_crew("The amazing spider-man")

TF list :the amazing spider-man 2
TF list :spider-man 3
TF list :hook
TF list :spider-man
TF list :charade
CV list: the amazing spider-man 2
CV list: spider-man
CV list: spider-man 3
CV list: hook
CV list: spider-man: no way home


## Feature & Vectorizer Comparison

Tested: {CountVectorizer, TF-IDF} × {with crew, without crew} on **the amazing spiderman**.

**With crew (CountVectorizer):**
the amazing spider-man 2
spider-man
spider-man 2
spider-man 3
spider-man

**With crew (TF-IDF):**
the amazing spider-man 2
spider-man 2
spider-man
spider-man 3
spider-man

**Without crew:**
TF list :the amazing spider-man 2
TF list :spider-man 3
TF list :hook
TF list :spider-man
TF list :charade
CV list: the amazing spider-man 2
CV list: spider-man
CV list: spider-man 3
CV list: hook
CV list: spider-man: no way home


**Findings:**
- Without crew, both TF-IDF and CV still surface mostly correct
  Spider-Man films, but each list includes one clear outlier
  (`hook`, `charade`) that crew-included results avoid entirely.
  Crew acts as a disambiguating signal — likely picking up shared
  actors/directors across franchise entries, that pure plot/genre
  text alone doesn't fully capture.
- CountVectorizer vs TF-IDF (with crew) produce nearly identical
  recommendations, differing mainly in ordering rather than
  relevance.

**Final approach:** TF-IDF on `overview + genre + crew` : since CV and
TF-IDF perform equivalently, TF-IDF is preferred as the more standard
choice for text similarity tasks.

In [30]:
pickle.dump(movie_ds,open('movie.pkl','wb'))
pickle.dump(tfidf_matrix, open('tfidf_matrix.pkl', 'wb'))